<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/07-convolutional-networks-vision-backbones.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Convolutional Neural Networks and Modern Vision Backbones** {#convolutional-neural-networks-modern-vision-backbones}

Chapter 06 treated architecture as one part of a generalization pipeline. This chapter looks inside the architectural assumption that made deep vision practical: nearby pixels form local patterns, the same pattern can occur at many positions, and useful features can be organized from edges to parts to objects. A convolutional neural network (CNN) encodes these assumptions through **local connectivity**, **weight sharing**, and a hierarchy of spatial feature maps.

The chapter continues the same UCI/scikit-learn Digits task used in Chapter 06. Reusing the fixed 60/20/20 split makes architecture the principal changing variable. The images are only $8\times8$, so the models are intentionally compact; an ImageNet-scale stem that downsamples by $32\times$ would erase them. This is itself an architectural lesson: a backbone must respect input resolution and task geometry.

Dataset sources: [scikit-learn `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) and [UCI Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B) (CC BY 4.0, DOI `10.24432/C50P49`).

### **Why Images Need Spatial Inductive Bias** {#why-images-need-spatial-inductive-bias}

An $H\times W$ image with $C$ channels can be flattened into a vector of length $CHW$, but flattening discards the explicit statement that adjacent coordinates are related. A dense layer from $CHW$ inputs to $D$ outputs uses $D(CHW+1)$ parameters and assigns a separate weight to every position. A convolution instead learns a small kernel and reuses it across locations. A $K_h\times K_w$ convolution from $C_{in}$ to $C_{out}$ channels uses

$$
C_{out}\left(C_{in}K_hK_w+1\right)
$$

parameters, independent of image height and width.

Weight sharing produces **translation equivariance** away from boundaries: shifting the input shifts the feature map. Equivariance is not invariance. A classifier becomes less position-sensitive only after pooling, aggregation, augmentation, or a learned downstream decision. Padding, stride, finite boundaries, and absolute positional components also make equivariance approximate rather than perfect.

CNNs are therefore not universally superior to dense networks. Their advantage comes when locality and repeated patterns match the data. For tabular features with no meaningful neighborhood, the same bias can be harmful. For images with abundant data and compute, Vision Transformers can learn broader interactions with weaker locality assumptions, but CNNs remain strong when sample efficiency, latency, or deployment maturity matters.

<details>
<summary><strong>PyTorch: compare dense and convolutional bias on one Digits split</strong></summary>

```python
import math
import random
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=707):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
targets = torch.tensor(digits.target, dtype=torch.long)
indices = np.arange(len(targets))
dev_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=606, stratify=digits.target
)
train_idx, val_idx = train_test_split(
    dev_idx, test_size=0.25, random_state=606, stratify=digits.target[dev_idx]
)
x_train, y_train = images[train_idx], targets[train_idx]
x_val, y_val = images[val_idx], targets[val_idx]
x_test, y_test = images[test_idx], targets[test_idx]


def loader(x, y, shuffle=False, seed=707, batch_size=128):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle, generator=generator)


@torch.no_grad()
def evaluate(model, x=x_val, y=y_val):
    model.eval()
    logits = model(x)
    return {"loss": F.cross_entropy(logits, y).item(),
            "accuracy": (logits.argmax(1) == y).float().mean().item()}


def fit(model, epochs=25, lr=3e-3, seed=707):
    seed_everything(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader(x_train, y_train, shuffle=True, seed=seed):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
    return evaluate(model)


class DenseDigits(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.net(x)


class CompactCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(32, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


seed_everything(707)
dense = DenseDigits()
dense_result = fit(dense, seed=707)
seed_everything(707)
cnn = CompactCNN()
cnn_result = fit(cnn, seed=707)

parameter_count = lambda model: sum(p.numel() for p in model.parameters())
assert cnn(x_train[:8]).shape == (8, 10)
assert 0 <= cnn_result["accuracy"] <= 1
print({"dense": (parameter_count(dense), dense_result),
       "cnn": (parameter_count(cnn), cnn_result)})
```

</details>

This one run is not a universal benchmark. It establishes the chapter contract and shows how to compare parameterization and held-out behavior on identical data. Later examples inspect the mechanisms that create the CNN's bias.

### **Discrete Convolution and Cross-Correlation** {#discrete-convolution-cross-correlation}

Deep-learning libraries usually implement **cross-correlation** even when the operation is named convolution. For input $X\in\mathbb{R}^{C_{in}\times H\times W}$ and kernel $K\in\mathbb{R}^{C_{out}\times C_{in}\times K_h\times K_w}$,

$$
Y_{o,i,j}=b_o+
\sum_{c=1}^{C_{in}}
\sum_{u=0}^{K_h-1}
\sum_{v=0}^{K_w-1}
K_{o,c,u,v}\,X_{c,i+u,j+v}.
$$

Mathematical convolution flips the kernel in both spatial dimensions; cross-correlation does not. Because kernels are learned, either convention can represent the same family of filters after reparameterization. The distinction matters when importing a fixed analytic kernel or comparing against signal-processing formulas.

At each output location the kernel computes an inner product between its weights and a local patch. A large positive response means the patch aligns with the learned pattern; a negative response means opposite contrast; a response near zero means weak alignment. One output channel is one learned detector, not necessarily a human-named edge. Stacking layers lets later filters combine earlier local responses.

![A padded input, moving kernel, and output feature map show how local dot products are reused across positions.](assets/dl07-convolution-padding.svg){fig-align="center" width="74%" fig-alt="Diagram of a convolution kernel moving over a padded image to produce a feature map."}

*Image source: local educational diagram derived from the discrete cross-correlation equation above.*

<details>
<summary><strong>PyTorch: reproduce `conv2d` from patches of a real digit</strong></summary>

```python
digit = x_train[:1]
edge_kernel = torch.tensor([[[[-1.0, 0.0, 1.0],
                              [-1.0, 0.0, 1.0],
                              [-1.0, 0.0, 1.0]]]])

library_response = F.conv2d(digit, edge_kernel, padding=1)
padded = F.pad(digit, (1, 1, 1, 1))
manual_response = torch.empty_like(library_response)
for row in range(8):
    for column in range(8):
        patch = padded[0, 0, row:row + 3, column:column + 3]
        manual_response[0, 0, row, column] = (patch * edge_kernel[0, 0]).sum()

flipped_response = F.conv2d(digit, edge_kernel.flip(-1, -2), padding=1)
assert torch.allclose(manual_response, library_response)
assert not torch.allclose(flipped_response, library_response)
print("strongest vertical-edge response:", library_response.abs().max().item())
```

</details>

The manual loop is useful for semantics, while production kernels lower convolution to optimized tiled operations. The mathematical output remains a bank of local dot products even when the hardware implementation does not literally slide one patch at a time.

### **Channels, Kernels, Stride, Padding, and Dilation** {#channels-kernels-stride-padding-dilation}

For one spatial dimension, input size $H$, kernel size $K$, dilation $D$, padding $P$, and stride $S$ produce

$$
H_{out}=\left\lfloor
\frac{H+2P-D(K-1)-1}{S}+1
\right\rfloor.
$$

The effective kernel span is $K_{eff}=D(K-1)+1$. Apply the same formula to width. The output channel count is determined by the number of kernels, not this spatial equation.

- **Stride** advances the kernel by more than one pixel and therefore performs filtering plus subsampling. Without adequate low-pass behavior, high-frequency content aliases.
- **Padding** controls border coverage and output size. Zero padding introduces an artificial frame; reflection or replication padding encode different boundary assumptions.
- **Dilation** spaces kernel taps apart, increasing spatial coverage without increasing the number of weights, but repeated large dilation can create checkerboard-like blind spots.
- **Channels** are mixed by a standard kernel: every output channel sums contributions from all input channels. Grouped and depthwise convolutions deliberately restrict this mixing.

Tensor layout in PyTorch is `[B, C, H, W]`. Confusing channels-last image arrays `[B, H, W, C]` with this convention produces either an immediate shape error or, more dangerously, an incorrectly configured model.

<details>
<summary><strong>PyTorch: predict and verify feature-map shapes</strong></summary>

```python
batch = x_train[:12]
configurations = [
    {"kernel_size": 3, "stride": 1, "padding": 0, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "padding": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 2, "padding": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "padding": 2, "dilation": 2},
]

shape_records = []
for cfg in configurations:
    layer = nn.Conv2d(1, 6, bias=False, **cfg)
    output = layer(batch)
    k, s, p, d = cfg["kernel_size"], cfg["stride"], cfg["padding"], cfg["dilation"]
    expected = math.floor((8 + 2 * p - d * (k - 1) - 1) / s + 1)
    assert output.shape == (12, 6, expected, expected)
    shape_records.append((cfg, tuple(output.shape)))

print(shape_records)
```

</details>

Shape calculation is not bookkeeping after the architecture is designed; it is part of the design. Every reduction changes memory, compute, receptive field, and the granularity available to dense-prediction heads.

### **Receptive Fields and Feature Hierarchies** {#receptive-fields-feature-hierarchies}

The **theoretical receptive field** of a unit is the input region that can influence it. Track receptive-field size $r_l$ and input jump $j_l$ between adjacent units at layer $l$:

$$
j_l=j_{l-1}S_l,
\qquad
r_l=r_{l-1}+(K_l-1)D_lj_{l-1},
$$

with $r_0=j_0=1$. Two stride-one $3\times3$ convolutions give a $5\times5$ receptive field, while using fewer parameters and inserting a nonlinearity between them. A stride-two layer increases the jump, so every later kernel expands coverage faster.

The **effective receptive field** is usually smaller and concentrated near the center because paths contribute with unequal gradient magnitude. It depends on learned weights, nonlinear gates, normalization, and data. Therefore, a theoretical field covering an entire image does not prove that distant context is used strongly.

Feature hierarchy is the functional consequence of growing context: early layers can detect contrast and local orientation; middle layers combine motifs; deeper layers represent task-specific parts and configurations. This interpretation is useful but not guaranteed per channel. Representation probing and attribution are needed before assigning semantics to a feature map.

<details>
<summary><strong>PyTorch: reveal the receptive field by input gradients</strong></summary>

```python
probe = x_train[0:1].clone().requires_grad_(True)
stack = nn.Sequential(
    nn.Conv2d(1, 1, 3, padding=1, bias=False), nn.ReLU(),
    nn.Conv2d(1, 1, 3, padding=1, bias=False),
)
with torch.no_grad():
    for layer in stack:
        if isinstance(layer, nn.Conv2d):
            layer.weight.fill_(1.0)

feature = stack(probe)
feature[0, 0, 4, 4].backward()
support = probe.grad[0, 0].abs() > 0
rows, columns = support.nonzero(as_tuple=True)

height = int(rows.max() - rows.min() + 1)
width = int(columns.max() - columns.min() + 1)
assert (height, width) == (5, 5)
assert support.sum() == 25
print({"gradient support": (height, width), "nonzero pixels": int(support.sum())})
```

</details>

The weights are fixed positive values so ReLU does not randomly close paths. For a trained CNN, visualizing gradient magnitude rather than only nonzero support reveals the effective field and potential boundary artifacts.

### **Pooling and Resolution Changes** {#pooling-resolution-changes}

Pooling aggregates a local neighborhood without learning a full channel-mixing kernel. Max pooling preserves the strongest activation,

$$
y_{c,i,j}=\max_{(u,v)\in\mathcal{N}_{i,j}}x_{c,u,v},
$$

while average pooling preserves the local mean. Max pooling is useful when feature presence matters more than exact location, but it routes gradient only through winning elements and can amplify isolated noise. Average pooling distributes gradients but may blur a sparse discriminative response.

Resolution reduction provides three benefits: larger downstream receptive fields, lower activation memory, and lower compute. The cost is irreversible spatial information loss. Modern networks often use strided convolutions so the downsampling filter is learned. Detection and segmentation retain multiple resolutions because a single low-resolution map is insufficient for small objects and precise boundaries.

Global average pooling maps `[B,C,H,W]` to `[B,C]` by averaging all locations. Compared with flattening followed by a large dense layer, it reduces parameters and encourages each channel to act as an image-level evidence map. It also discards explicit layout, so it is inappropriate when the output itself must remain spatial.

<details>
<summary><strong>PyTorch: compare information and gradient routing through pooling</strong></summary>

```python
pool_input = x_train[:4].clone().requires_grad_(True)
max_output = F.max_pool2d(pool_input, kernel_size=2, stride=2)
max_output.sum().backward(retain_graph=True)
max_nonzero_gradients = int((pool_input.grad != 0).sum())

pool_input.grad.zero_()
average_output = F.avg_pool2d(pool_input, kernel_size=2, stride=2)
average_output.sum().backward()
average_nonzero_gradients = int((pool_input.grad != 0).sum())

assert max_output.shape == average_output.shape == (4, 1, 4, 4)
assert max_nonzero_gradients <= 4 * 1 * 4 * 4
assert average_nonzero_gradients >= max_nonzero_gradients
print({"max gradient locations": max_nonzero_gradients,
       "average gradient locations": average_nonzero_gradients})
```

</details>

Pooling is neither mandatory nor harmless. Choose the resolution schedule from the output geometry and memory budget, then verify small-feature performance instead of assuming that downsampling automatically creates useful invariance.

### **Depthwise and Separable Convolutions** {#depthwise-separable-convolutions}

A standard convolution simultaneously mixes space and channels. With $C_{in}$ input channels, $C_{out}$ output channels, and a $K\times K$ kernel, its weight count and per-location multiply-accumulate count are both proportional to

$$
K^2C_{in}C_{out}.
$$

A **depthwise separable convolution** factorizes this operation:

1. a depthwise $K\times K$ convolution applies one spatial filter per input channel, costing $K^2C_{in}$;
2. a pointwise $1\times1$ convolution mixes channels, costing $C_{in}C_{out}$.

The ratio relative to a standard convolution is

$$
\frac{K^2C_{in}+C_{in}C_{out}}{K^2C_{in}C_{out}}
=\frac{1}{C_{out}}+\frac{1}{K^2}.
$$

This large arithmetic reduction powers MobileNet-like models. It is a structural constraint, however: spatial filtering cannot immediately use arbitrary cross-channel combinations. Hardware speedup may also be smaller than the FLOP reduction because memory movement, kernel launch overhead, and implementation quality matter.

![Depthwise convolution assigns a dedicated spatial kernel to each input channel before pointwise channel mixing.](assets/dl07-depthwise-separable.svg){fig-align="center" width="72%" fig-alt="Depthwise separable convolution diagram showing independent spatial filtering followed by channel mixing."}

*Image source: local educational diagram created after reviewing [Dive into Deep Learning Compiler, Depthwise Convolution](https://tvm.d2l.ai/chapter_common_operators/depthwise_conv.html) and the parameter factorization above.*

<details>
<summary><strong>PyTorch: compare factorization on Digits feature maps</strong></summary>

```python
features = nn.Conv2d(1, 32, 3, padding=1)(x_train[:16])
standard = nn.Conv2d(32, 64, 3, padding=1, bias=False)
separable = nn.Sequential(
    nn.Conv2d(32, 32, 3, padding=1, groups=32, bias=False),
    nn.Conv2d(32, 64, 1, bias=False),
)

standard_output = standard(features)
separable_output = separable(features)
standard_parameters = sum(p.numel() for p in standard.parameters())
separable_parameters = sum(p.numel() for p in separable.parameters())
theoretical = 3 * 3 * 32 + 32 * 64

assert standard_output.shape == separable_output.shape == (16, 64, 8, 8)
assert separable_parameters == theoretical
assert separable_parameters < standard_parameters
print({"standard": standard_parameters, "depthwise_separable": separable_parameters,
       "parameter ratio": separable_parameters / standard_parameters})
```

</details>

The outputs share a shape but are not numerically equivalent. Factorization defines a different hypothesis class; it is selected because the efficiency-accuracy trade-off is favorable, not because it is an algebraic identity for arbitrary standard kernels.

### **From LeNet and AlexNet to VGG** {#lenet-alexnet-vgg}

Historical architectures are best understood as changes in the **design vocabulary**, not as a list of layer counts.

- **LeNet-5** established the convolution-pooling-classifier pattern for small character images. Its local receptive fields and shared weights replaced fully connected processing of every pixel.
- **AlexNet** demonstrated that deeper CNNs, ReLU, dropout, data augmentation, and GPU training could scale image recognition. Large early kernels and dense classifier layers reflect the hardware and dataset regime of its time.
- **VGG** made depth systematic by repeatedly stacking $3\times3$ convolutions and doubling channels after resolution reductions. Two $3\times3$ layers obtain a $5\times5$ receptive field with two nonlinearities and usually fewer weights than one dense $5\times5$ layer.

These ImageNet architectures cannot be copied literally onto $8\times8$ Digits. Repeated pooling would collapse spatial dimensions, and a large fully connected head would dominate parameter count. The reusable lesson is the stage pattern: maintain resolution within a stage, then trade spatial size for channel capacity.

<details>
<summary><strong>PyTorch: adapt three historical design patterns to $8\times8$ inputs</strong></summary>

```python
class TinyLeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 3, padding=1), nn.Tanh(), nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 3, padding=1), nn.Tanh(), nn.AvgPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 2 * 2, 32), nn.Tanh(), nn.Linear(32, 10))

    def forward(self, x):
        return self.head(self.features(x))


class TinyAlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 24, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(48, 10))

    def forward(self, x):
        return self.head(self.features(x))


class TinyVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(32, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


historical = [TinyLeNet(), TinyAlexNet(), TinyVGG()]
records = []
for model in historical:
    output = model(x_train[:8])
    records.append((model.__class__.__name__, sum(p.numel() for p in model.parameters()), tuple(output.shape)))
    assert output.shape == (8, 10)
print(records)
```

</details>

The classes preserve characteristic motifs, not published benchmark models. Faithful reproduction requires original input size, preprocessing, initialization, optimizer, and training recipe; architecture names alone do not recreate historical results.

### **Residual Networks** {#residual-networks}

As plain networks deepen, optimization can degrade even when the deeper model can theoretically represent the shallower one. A residual block learns a residual function $F$ around an identity path:

$$
h_{l+1}=h_l+F(h_l;\theta_l).
$$

The backward signal contains a direct term

$$
\frac{\partial L}{\partial h_l}
=\frac{\partial L}{\partial h_{l+1}}
\left(I+\frac{\partial F}{\partial h_l}\right),
$$

so the gradient need not pass exclusively through every transformation. This does not guarantee perfect conditioning, but it makes identity-like behavior easy and supports much deeper optimization.

Addition requires matching shapes. When resolution or channel count changes, a projection shortcut such as a stride-two $1\times1$ convolution aligns them. The placement of normalization and activation also matters. In pre-activation ResNets, normalization and activation precede convolution, leaving a cleaner identity route.

![A residual unit routes the input through both an identity shortcut and a learned residual branch before addition.](assets/dl07-residual-unit.svg){fig-align="center" width="68%" fig-alt="Residual block diagram with identity shortcut and learned branch merged by addition."}

*Image source: local educational diagram derived from the residual mapping equation and [He et al., Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385).*

<details>
<summary><strong>PyTorch: train a compact residual network on Digits</strong></summary>

```python
class BasicResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.relu(x + self.branch(x))


class TinyResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 24, 3, padding=1), nn.ReLU())
        self.blocks = nn.Sequential(BasicResidualBlock(24), BasicResidualBlock(24), BasicResidualBlock(24))
        self.head = nn.Linear(24, 10)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.head(x.mean((-2, -1)))


seed_everything(708)
resnet = TinyResNet()
resnet_validation = fit(resnet, epochs=25, lr=2e-3, seed=708)
resnet_test = evaluate(resnet, x_test, y_test)
assert resnet(x_train[:5]).shape == (5, 10)
assert resnet_validation["accuracy"] > 0.80
print({"validation": resnet_validation, "test": resnet_test})
```

</details>

Residual connections solve an optimization interface problem; they do not replace data quality, regularization, or a suitable resolution schedule. If a residual network fails, inspect activation/gradient statistics, normalization mode, and projection shapes before assuming that depth itself is the cause.

### **EfficientNet and ConvNeXt** {#efficientnet-convnext}

EfficientNet and ConvNeXt represent two different routes to strong modern CNNs.

**EfficientNet** begins from an efficient mobile block and scales depth, width, and input resolution together. A simplified compound rule is

$$
d=\alpha^\phi,\qquad w=\beta^\phi,\qquad r=\gamma^\phi,
\qquad \alpha\beta^2\gamma^2\approx2,
$$

where $\phi$ is a global scale coefficient. The exponents reflect approximate convolutional cost: width influences both input and output channels, while resolution affects two spatial axes. EfficientNet blocks commonly combine expansion, depthwise convolution, squeeze-and-excitation, projection, and residual paths.

**ConvNeXt** modernizes a ResNet using design choices informed by Transformers while remaining convolutional: patch-like downsampling stems, large-kernel depthwise convolutions, fewer activation/normalization sites, LayerNorm, inverted bottlenecks, GELU, and stage ratios adjusted toward later processing. Its lesson is not that one operator won; training recipes and system-level block design can make a mature inductive bias competitive.

FLOPs are only a proxy for latency. Depthwise layers can be memory-bound, larger activations increase memory traffic, and a theoretically efficient block may map poorly to a target accelerator. Measure throughput, peak memory, and tail latency at the deployment batch size and precision.

<details>
<summary><strong>PyTorch: compare mobile and ConvNeXt-style blocks on Digits features</strong></summary>

```python
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.SiLU(),
            nn.Conv2d(hidden, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.net(x)


class MBConv(nn.Module):
    def __init__(self, channels=24, expansion=4):
        super().__init__()
        hidden = channels * expansion
        self.branch = nn.Sequential(
            nn.Conv2d(channels, hidden, 1), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden), nn.SiLU(),
            SqueezeExcitation(hidden), nn.Conv2d(hidden, channels, 1),
        )

    def forward(self, x):
        return x + self.branch(x)


class ConvNeXtBlock(nn.Module):
    def __init__(self, channels=24, expansion=4):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, 7, padding=3, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.expand = nn.Linear(channels, channels * expansion)
        self.contract = nn.Linear(channels * expansion, channels)

    def forward(self, x):
        residual = x
        x = self.depthwise(x).permute(0, 2, 3, 1)
        x = self.contract(F.gelu(self.expand(self.norm(x))))
        return residual + x.permute(0, 3, 1, 2)


stem_features = nn.Conv2d(1, 24, 3, padding=1)(x_train[:16])
mobile_block, convnext_block = MBConv(), ConvNeXtBlock()
mobile_output = mobile_block(stem_features)
convnext_output = convnext_block(stem_features)

count = lambda module: sum(p.numel() for p in module.parameters())
assert mobile_output.shape == convnext_output.shape == stem_features.shape
print({"MBConv parameters": count(mobile_block),
       "ConvNeXt-style parameters": count(convnext_block)})
```

</details>

These are component-level implementations, not full EfficientNet or ConvNeXt reproductions. They expose the data path and tensor layout while leaving large-scale recipes and optimized kernels to established libraries such as torchvision.

### **U-Net, Feature Pyramids, Detection, and Segmentation** {#unet-feature-pyramids-detection-segmentation}

Classification maps an image to one label; detection and segmentation must preserve or reconstruct spatial correspondence. A backbone that repeatedly downsamples gains semantics and context but loses boundary detail.

**U-Net** resolves this with an encoder-decoder. The encoder produces progressively lower-resolution features; the decoder upsamples them. Skip connections concatenate encoder features at matching resolutions so the decoder receives both semantic context and fine detail. Concatenation differs from residual addition: it preserves both tensors as separate channels and lets a later convolution decide how to combine them.

A **Feature Pyramid Network (FPN)** instead constructs semantically strong feature maps at several resolutions through a top-down pathway and lateral $1\times1$ projections. Detection heads can use high-resolution pyramid levels for small objects and low-resolution levels for large objects. The core distinction is task interface: U-Net commonly decodes a dense output, while FPN exposes a reusable multi-scale representation to heads.

Detection additionally predicts class and location. Anchor-based heads score predefined boxes and regress offsets; anchor-free heads predict centers, corners, or distances directly. Segmentation predicts per-pixel class distributions. Their losses, assignment rules, and evaluation metrics differ, but all depend on retaining spatial hierarchy.

The Digits labels do not contain segmentation masks. For a mechanism-level exercise, pixels greater than zero define a deterministic **foreground proxy**. It is not a research segmentation benchmark; it lets the same source images verify skip shapes and dense output without inventing another dataset.

<details>
<summary><strong>PyTorch: build a tiny U-Net path for Digits foreground masks</strong></summary>

```python
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(8, 16, 3, padding=1), nn.ReLU())
        self.decode = nn.Sequential(nn.Conv2d(24, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 1, 1))

    def forward(self, x):
        high_resolution = self.enc1(x)                         # [B, 8, 8, 8]
        low_resolution = self.enc2(F.max_pool2d(high_resolution, 2))  # [B, 16, 4, 4]
        upsampled = F.interpolate(low_resolution, size=high_resolution.shape[-2:], mode="bilinear", align_corners=False)
        logits = self.decode(torch.cat([high_resolution, upsampled], dim=1))
        return logits, (high_resolution, low_resolution, upsampled)


segmenter = TinyUNet()
digit_batch = x_train[:32]
foreground = (digit_batch > 0).float()
mask_logits, pyramid = segmenter(digit_batch)
mask_loss = F.binary_cross_entropy_with_logits(mask_logits, foreground)
mask_loss.backward()

assert mask_logits.shape == foreground.shape == (32, 1, 8, 8)
assert [tuple(t.shape[-2:]) for t in pyramid] == [(8, 8), (4, 4), (8, 8)]
assert all(parameter.grad is not None for parameter in segmenter.parameters())
print({"mask loss": mask_loss.item(), "feature shapes": [tuple(t.shape) for t in pyramid]})
```

</details>

The example verifies topology, not segmentation quality. A valid application requires human- or process-derived masks, split-by-entity evaluation, class-imbalance handling, and metrics such as IoU or Dice that reflect spatial overlap.

### **CNNs vs Vision Transformers** {#cnns-vs-vision-transformers}

A CNN aggregates local neighborhoods with shared kernels. A Vision Transformer (ViT) divides an image into patches, embeds each patch as a token, adds position information, and uses self-attention so each token can interact with every other token.

For patch size $P$, an $H\times W$ image becomes

$$
N=\frac{H}{P}\frac{W}{P}
$$

tokens. Full self-attention has $O(N^2D)$ interaction cost for embedding dimension $D$, while a fixed-kernel convolution has cost linear in spatial positions for fixed channels and kernel size. ViT's global interaction is flexible, but it usually relies more heavily on data scale, pretraining, augmentation, and regularization. Hybrid and hierarchical Transformers reintroduce locality or multiscale structure because dense global attention is not the only useful image prior.

The comparison should include the training recipe and deployment environment. A pretrained ViT may transfer better than a CNN trained from scratch; a compact CNN may be faster at batch size one; windowed attention may scale differently from global attention. “CNN versus Transformer” is not one operator-level contest.

<details>
<summary><strong>PyTorch: turn the same $8\times8$ digits into image tokens</strong></summary>

```python
class TinyVisionTransformer(nn.Module):
    def __init__(self, patch_size=2, dimension=32, heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.patch_projection = nn.Linear(patch_size * patch_size, dimension)
        self.class_token = nn.Parameter(torch.zeros(1, 1, dimension))
        self.position = nn.Parameter(torch.zeros(1, 17, dimension))  # 16 patches + class token
        layer = nn.TransformerEncoderLayer(
            d_model=dimension, nhead=heads, dim_feedforward=64,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(dimension, 10)

    def forward(self, x):
        patches = F.unfold(x, kernel_size=self.patch_size, stride=self.patch_size).transpose(1, 2)
        tokens = self.patch_projection(patches)
        cls = self.class_token.expand(x.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.position[:, :tokens.shape[1] + 1]
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0]), patches


vit = TinyVisionTransformer()
vit_logits, digit_patches = vit(x_train[:8])
cnn_logits = cnn(x_train[:8])

assert digit_patches.shape == (8, 16, 4)
assert vit_logits.shape == cnn_logits.shape == (8, 10)
print({"patch sequence": tuple(digit_patches.shape),
       "CNN parameters": sum(p.numel() for p in cnn.parameters()),
       "ViT parameters": sum(p.numel() for p in vit.parameters())})
```

</details>

On $8\times8$ inputs, sixteen $2\times2$ patches make the tokenization visible. A serious comparison would match parameter count, augmentation, search budget, pretraining, and compute, then repeat seeds. The example establishes representation geometry rather than declaring a winner.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

CNN design is a sequence of decisions about **where information may interact, how resolution changes, and what task interface the backbone exposes**. Individual layer names matter less than the resulting receptive field, tensor hierarchy, compute pattern, and information loss.

| Mechanism or family | Main inductive bias | Efficiency advantage | Important limitation |
|---|---|---|---|
| dense image MLP | no explicit spatial locality | simple implementation | position-specific weights and weak sample efficiency |
| standard convolution | local shared patterns and channel mixing | linear spatial scaling for fixed kernel | local context grows only through depth/dilation |
| stride/pooling | coarser spatial summaries | lower activation memory and compute | aliasing and irreversible detail loss |
| dilated convolution | sparse wider context | larger field without more weights | gridding artifacts and unchanged activation size |
| depthwise separable convolution | per-channel spatial filtering then mixing | large parameter/FLOP reduction | constrained interaction and hardware sensitivity |
| VGG-style stages | repeated small kernels | regular hierarchy and multiple nonlinearities | expensive activations and no shortcut path |
| residual networks | learn changes around identity | stable deep optimization | shape alignment and normalization still matter |
| EfficientNet/MBConv | balanced scaling and mobile factorization | favorable accuracy-efficiency trade-off | recipe and device-specific latency matter |
| ConvNeXt | modernized convolutional block design | competitive mature kernels | large kernels and layouts require measurement |
| U-Net/FPN | multi-resolution fusion | recovers detail for dense tasks | memory cost from feature retention and fusion |
| Vision Transformer | patch tokens with content-dependent global mixing | flexible long-range interaction | quadratic global attention and weaker locality bias |

The shared Digits study creates a continuous path:

1. Preserve `[B,C,H,W]` instead of flattening away neighborhood structure.
2. Interpret convolution as shared local cross-correlation and verify its exact indexing.
3. Calculate every output shape before implementation; stride and padding change both geometry and cost.
4. Track receptive field and feature resolution together because context and detail trade against each other.
5. Use factorized, residual, or modernized blocks for a stated efficiency or optimization reason.
6. Adapt architecture scale to input resolution rather than copying an ImageNet topology by name.
7. Expose multi-resolution features when the output is spatial, and evaluate the task with spatial metrics.
8. Compare CNNs and Transformers under matched data, recipes, compute, and deployment constraints.

The chapter's code uses real digit images in every example: manual cross-correlation operates on an observed sample, receptive fields are probed through its pixels, efficient blocks process its feature maps, and the encoder-decoder reconstructs its foreground. This continuity makes tensor shapes and architectural assumptions concrete. Chapter 08 changes the data geometry from two-dimensional space to ordered time, where recurrence, temporal convolution, attention, and state-space models make different trade-offs about memory and parallelism.